In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
from tqdm import tqdm

from shapely.geometry import Polygon

Load census variables and parse for relevant information

In [2]:
# Create mapping for characteristic IDs to column names
characteristic_mapping = {
    1: 'CHAR_POP21'  # Population, 2021
}

df_cen_cma_data = pd.read_csv('../data/census/98-401-X2021002_eng_CSV/98-401-X2021002_English_CSV_data.csv', encoding='latin')
df_cen_cma_data = df_cen_cma_data[['DGUID', 'GEO_LEVEL', 'GEO_NAME', 'CHARACTERISTIC_ID', 'C1_COUNT_TOTAL']]

# Filter for only the characteristics we want
df_cen_cma_data = df_cen_cma_data[df_cen_cma_data['CHARACTERISTIC_ID'].isin(characteristic_mapping.keys())]

# Map characteristic IDs to column names
df_cen_cma_data['CHARACTERISTIC_COLUMN'] = df_cen_cma_data['CHARACTERISTIC_ID'].map(characteristic_mapping)

# Pivot to create separate columns for each characteristic
df_cen_cma_data = df_cen_cma_data.pivot_table(
    index=['DGUID', 'GEO_LEVEL', 'GEO_NAME'], 
    columns='CHARACTERISTIC_COLUMN', 
    values='C1_COUNT_TOTAL',
    aggfunc='first'
).reset_index()

# Flatten column names
df_cen_cma_data.columns.name = None

In [3]:
df_ada_cma_rel = pd.read_csv('../data/census/ada_cma_relation.csv')

In [4]:
gdf_cma = gpd.read_file('../data/census/lcma000b21a_e')
gdf_cma = gdf_cma[['CMAUID', 'DGUID', 'CMANAME', 'PRUID', 'geometry']]

Load tariff information and join to CMAs

In [5]:
# Load ADA-level tariff counts and percents
df_tariffs_ada_count = pd.read_excel('tariff-impacts-ada-data.xlsx', sheet_name='Counts')
df_tariffs_ada_pct = pd.read_excel('tariff-impacts-ada-data.xlsx', sheet_name='Percents')

In [6]:
# Prepare the relation dataframe
df_ada_cma_rel_clean = df_ada_cma_rel[['CMADGUID_RMRIDUGD']].rename(columns={'CMADGUID_RMRIDUGD': 'CMADGUID'})
df_ada_cma_rel_clean['ADADGUID'] = df_ada_cma_rel['ADADGUID_ADAIDUGD']

# Clean counts dataframe (drop geometry)
df_tariffs_ada_count_clean = df_tariffs_ada_count.drop(columns=['geometry'])

# Join ADA counts with CMA relation
df_tariffs_count_joined = df_tariffs_ada_count_clean.merge(df_ada_cma_rel_clean, on='ADADGUID', how='left')

# Filter out ADAs not in any CMA
df_tariffs_count_filtered = df_tariffs_count_joined.dropna(subset=['CMADGUID'])

# Group by CMADGUID and sum all tariff columns
tariff_columns_count = [col for col in df_tariffs_count_filtered.columns if col not in ['ADADGUID', 'CMADGUID']]
df_tariffs_cma_count = df_tariffs_count_filtered.groupby('CMADGUID')[tariff_columns_count].sum().reset_index()

print(f"Shape of CMA tariffs (counts) dataframe: {df_tariffs_cma_count.shape}")
df_tariffs_cma_count.head()

Shape of CMA tariffs (counts) dataframe: (152, 25)


,CMADGUID,Auto_B,Alum_B,Steel_B,Cop_B,Lum_B,Ene_B,CUSMA_B,Total_B,Auto_E,...,CUSMA_E,Total_E,Auto_C,Alum_C,Steel_C,Cop_C,Lum_C,Ene_C,CUSMA_C,Total_C
0,2021S0503001,7,17,14,2,8,10,118,118,40,...,1966,1966,146,226,216,23,110,247,2667,2667
1,2021S0503205,35,89,56,10,20,39,400,415,660,...,5596,6414,1252,2841,1673,137,676,1498,10111,11040
2,2021S0503305,15,37,34,5,8,19,177,191,316,...,3539,3919,378,684,598,102,344,387,4466,4725
3,2021S0503310,5,17,13,1,8,9,136,142,153,...,2460,3487,313,620,1401,22,365,1168,3691,4746
4,2021S0503320,9,26,15,0,9,10,126,132,172,...,1530,1696,191,389,309,1,215,223,2205,2357


In [7]:
# Map percent columns (_1, _2, _3) to count columns (_B, _E, _C)
percent_to_count_map = {
    '_1': '_B',
    '_2': '_E',
    '_3': '_C'
}

# Melt both dataframes to long format for easier weighting
df_pct_long = df_tariffs_ada_pct.melt(id_vars=['ADADGUID'], var_name='tariff_pct', value_name='percent')
df_count_long = df_tariffs_ada_count_clean.melt(id_vars=['ADADGUID'], var_name='tariff_count', value_name='count')

# Extract base name and type
df_pct_long['base'] = df_pct_long['tariff_pct'].str.replace(r'_(1|2|3)$', '', regex=True)
df_pct_long['suffix'] = df_pct_long['tariff_pct'].str.extract(r'_(1|2|3)$')[0]

df_count_long['base'] = df_count_long['tariff_count'].str.replace(r'_[BEC]$', '', regex=True)
df_count_long['suffix'] = df_count_long['tariff_count'].str.extract(r'_([BEC])$')[0]

# Map percent suffixes (1,2,3) → B,E,C
suffix_map = {'1': 'B', '2': 'E', '3': 'C'}
df_pct_long['suffix'] = df_pct_long['suffix'].map(suffix_map)

# Merge percents with counts (aligned by ADADGUID + tariff type)
df_merge_long = df_pct_long.merge(
    df_count_long,
    left_on=['ADADGUID', 'base', 'suffix'],
    right_on=['ADADGUID', 'base', 'suffix'],
    how='left'
)

# Compute weighted contribution = percent * count
df_merge_long['weighted'] = df_merge_long['percent'] * df_merge_long['count']

# Add CMA ID
df_merge_long = df_merge_long.merge(df_ada_cma_rel_clean, on='ADADGUID', how='left')

# Drop ADAs not in any CMA
df_merge_long = df_merge_long.dropna(subset=['CMADGUID'])

# Compute CMA-level weighted averages
df_pct_cma = (
    df_merge_long.groupby(['CMADGUID', 'base', 'suffix'])
    .apply(lambda g: g['weighted'].sum() / g['count'].sum() if g['count'].sum() > 0 else np.nan)
    .reset_index(name='cma_percent')
)

# Reconstruct wide format
df_pct_cma['colname'] = df_pct_cma['base'] + '_' + df_pct_cma['suffix']
df_tariffs_cma_pct = df_pct_cma.pivot(index='CMADGUID', columns='colname', values='cma_percent').reset_index()

print(f"Shape of CMA tariffs (percents) dataframe: {df_tariffs_cma_pct.shape}")
df_tariffs_cma_pct.head()

Shape of CMA tariffs (percents) dataframe: (152, 25)


/tmp/ipykernel_2091392/2760415326.py:43: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g['weighted'].sum() / g['count'].sum() if g['count'].sum() > 0 else np.nan)


colname,CMADGUID,Alum_B,Alum_C,Alum_E,Auto_B,Auto_C,Auto_E,CUSMA_B,CUSMA_C,CUSMA_E,...,Ene_E,Lum_B,Lum_C,Lum_E,Steel_B,Steel_C,Steel_E,Total_B,Total_C,Total_E
0,2021S0503001,0.005691,0.002238,0.007065,0.005103,0.001510,0.002101,0.021111,0.025727,0.076244,...,0.001553,0.004841,0.001141,0.004708,0.007445,0.002153,0.008430,0.021111,0.025727,0.076244
1,2021S0503205,0.012824,0.012252,0.032155,0.006093,0.005186,0.012695,0.060359,0.043152,0.056219,...,0.025716,0.009039,0.009175,0.017713,0.009748,0.007021,0.016886,0.061639,0.046423,0.060594
2,2021S0503305,0.011629,0.008997,0.025271,0.006637,0.005317,0.008496,0.055809,0.061789,0.079371,...,0.027168,0.005070,0.004787,0.022372,0.015260,0.008183,0.024982,0.060623,0.064767,0.089119
3,2021S0503310,0.010659,0.011093,0.027720,0.009016,0.006249,0.033448,0.108481,0.070211,0.058331,...,0.082831,0.017934,0.007487,0.013391,0.012512,0.022146,0.081231,0.107721,0.082464,0.091723
4,2021S0503320,0.013900,0.008215,0.020012,0.006885,0.004083,0.026298,0.065049,0.052572,0.155038,...,0.021174,0.023556,0.011265,0.020492,0.016517,0.006948,0.025372,0.066470,0.055045,0.153233


In [8]:
# Rename DGUID to CMADGUID in census data
df_cen_cma_data_renamed = df_cen_cma_data.rename(columns={'DGUID': 'CMADGUID'})

# Merge census with CMA counts
df_final_counts = df_cen_cma_data_renamed.merge(df_tariffs_cma_count, on='CMADGUID', how='inner')

# Merge census with CMA percents
df_final_percents = df_cen_cma_data_renamed.merge(df_tariffs_cma_pct, on='CMADGUID', how='inner')

print(f"Shape of final counts dataframe: {df_final_counts.shape}")
print(f"Shape of final percents dataframe: {df_final_percents.shape}")

df_final_counts.head()
df_final_percents.head()

Shape of final counts dataframe: (152, 28)
Shape of final percents dataframe: (152, 28)


,CMADGUID,GEO_LEVEL,GEO_NAME,CHAR_POP21,Alum_B,Alum_C,Alum_E,Auto_B,Auto_C,Auto_E,...,Ene_E,Lum_B,Lum_C,Lum_E,Steel_B,Steel_C,Steel_E,Total_B,Total_C,Total_E
0,2021S0503001,Census metropolitan area,St. John's,212579.0,0.005691,0.002238,0.007065,0.005103,0.001510,0.002101,...,0.001553,0.004841,0.001141,0.004708,0.007445,0.002153,0.008430,0.021111,0.025727,0.076244
1,2021S0503205,Census metropolitan area,Halifax,465703.0,0.012824,0.012252,0.032155,0.006093,0.005186,0.012695,...,0.025716,0.009039,0.009175,0.017713,0.009748,0.007021,0.016886,0.061639,0.046423,0.060594
2,2021S0503305,Census metropolitan area,Moncton,157717.0,0.011629,0.008997,0.025271,0.006637,0.005317,0.008496,...,0.027168,0.005070,0.004787,0.022372,0.015260,0.008183,0.024982,0.060623,0.064767,0.089119
3,2021S0503310,Census metropolitan area,Saint John,130613.0,0.010659,0.011093,0.027720,0.009016,0.006249,0.033448,...,0.082831,0.017934,0.007487,0.013391,0.012512,0.022146,0.081231,0.107721,0.082464,0.091723
4,2021S0503320,Census metropolitan area,Fredericton,108610.0,0.013900,0.008215,0.020012,0.006885,0.004083,0.026298,...,0.021174,0.023556,0.011265,0.020492,0.016517,0.006948,0.025372,0.066470,0.055045,0.153233


In [9]:
# Prepare geometry data
gdf_cma_geom = gdf_cma[['DGUID', 'geometry']].rename(columns={'DGUID': 'CMADGUID'})

# ---- COUNTS ----
gdf_final_counts_full = df_final_counts.merge(gdf_cma_geom, on='CMADGUID', how='inner')
gdf_final_counts_full = gpd.GeoDataFrame(gdf_final_counts_full, geometry='geometry')

gdf_cma_centroids = gdf_cma_geom.copy()
gdf_cma_centroids['geometry'] = gdf_cma_centroids['geometry'].centroid
gdf_cma_centroids = gdf_cma_centroids.set_crs(gdf_cma.crs).to_crs('EPSG:4326')

gdf_final_counts_centroids = df_final_counts.merge(gdf_cma_centroids, on='CMADGUID', how='inner')
gdf_final_counts_centroids = gpd.GeoDataFrame(gdf_final_counts_centroids, geometry='geometry', crs='EPSG:4326')

gdf_final_counts_full.to_file('../data/cma/cma_tariffs_counts_full_geometry.gpkg', driver='GPKG')
df_counts_centroids_csv = gdf_final_counts_centroids.copy()
df_counts_centroids_csv['geometry'] = gdf_final_counts_centroids.geometry.apply(lambda geom: geom.wkt)
df_counts_centroids_csv.to_csv('../data/cma/cma_tariffs_counts_centroids.csv', index=False)

# ---- PERCENTS ----
gdf_final_percents_full = df_final_percents.merge(gdf_cma_geom, on='CMADGUID', how='inner')
gdf_final_percents_full = gpd.GeoDataFrame(gdf_final_percents_full, geometry='geometry')

gdf_final_percents_centroids = df_final_percents.merge(gdf_cma_centroids, on='CMADGUID', how='inner')
gdf_final_percents_centroids = gpd.GeoDataFrame(gdf_final_percents_centroids, geometry='geometry', crs='EPSG:4326')

gdf_final_percents_full.to_file('../data/cma/cma_tariffs_percents_full_geometry.gpkg', driver='GPKG')
df_percents_centroids_csv = gdf_final_percents_centroids.copy()
df_percents_centroids_csv['geometry'] = gdf_final_percents_centroids.geometry.apply(lambda geom: geom.wkt)
df_percents_centroids_csv.to_csv('../data/cma/cma_tariffs_percents_centroids.csv', index=False)

print(f"Counts full geometry shape: {gdf_final_counts_full.shape}")
print(f"Percents full geometry shape: {gdf_final_percents_full.shape}")
print(f"Counts centroids shape: {gdf_final_counts_centroids.shape}")
print(f"Percents centroids shape: {gdf_final_percents_centroids.shape}")

/tmp/ipykernel_2091392/312333940.py:17: UserWarning: Geometry column does not contain geometry.
  df_counts_centroids_csv['geometry'] = gdf_final_counts_centroids.geometry.apply(lambda geom: geom.wkt)


Counts full geometry shape: (156, 29)
Percents full geometry shape: (156, 29)
Counts centroids shape: (156, 29)
Percents centroids shape: (156, 29)


/tmp/ipykernel_2091392/312333940.py:29: UserWarning: Geometry column does not contain geometry.
  df_percents_centroids_csv['geometry'] = gdf_final_percents_centroids.geometry.apply(lambda geom: geom.wkt)
